In [ ]:
import os
import time
import random
import requests
import psycopg2

# ----------------------------
# CONFIG
# ----------------------------
API_KEY = os.getenv("BALLDONTLIE_API_KEY")
BASE_URL = "https://api.balldontlie.io/v1/games"

DB_CFG = dict(
    dbname="nba_data",
    user="postgres",
    password=os.getenv("NBA_DB_PASSWORD"),
    host="localhost",
    port=5432
)

SEASONS = [2023, 2024, 2025]   # 2025 = YTD
PER_PAGE = 50

THROTTLE_SEC_START = 4.0
THROTTLE_SEC_MAX   = 12.0

HEADERS = {"Authorization": API_KEY, "Accept": "application/json"}

# ----------------------------
# THROTTLE
# ----------------------------
throttle_sec = THROTTLE_SEC_START
last_request_ts = 0.0

def throttle_wait():
    global last_request_ts, throttle_sec
    now = time.time()
    elapsed = now - last_request_ts
    if elapsed < throttle_sec:
        time.sleep(throttle_sec - elapsed)
    last_request_ts = time.time()

def bump_throttle():
    global throttle_sec
    new_val = min(THROTTLE_SEC_MAX, throttle_sec * 1.35 + 0.25)
    if new_val > throttle_sec:
        throttle_sec = new_val
        print(f"  🛑 Rate limit hit → increasing throttle to {throttle_sec:.2f}s")

def safe_get(session, params, max_retries=25):
    for attempt in range(1, max_retries + 1):
        throttle_wait()
        r = session.get(BASE_URL, headers=HEADERS, params=params, timeout=30)

        if r.status_code == 429:
            bump_throttle()
            ra = r.headers.get("Retry-After")
            if ra:
                try:
                    wait = int(float(ra)) + 1
                except ValueError:
                    wait = min(25 * attempt, 180)
            else:
                wait = min(25 * attempt, 180)
            print(f"  ⚠️ HTTP 429. Retry {attempt}/{max_retries} in {wait}s")
            time.sleep(wait)
            continue

        if r.status_code in (500, 502, 503, 504):
            wait = min(2 ** attempt, 30) + random.uniform(0, 1.5)
            print(f"  ⚠️ HTTP {r.status_code}. Retry {attempt}/{max_retries} in {wait:.1f}s")
            time.sleep(wait)
            continue

        r.raise_for_status()
        return r

    raise RuntimeError(f"Failed after {max_retries} retries for params={params}")

def pretty_print_row(game_date, team, opp, site, pts_for, pts_against, margin, result):
    print(
        f"  👀 Sample inserted row:\n"
        f"    {game_date} | {team} vs {opp} | "
        f"Site={site} | 1H: {pts_for}–{pts_against} | "
        f"Margin={margin:+d} | Result={result}"
    )

# ----------------------------
# DB SETUP
# ----------------------------
print("Connecting to Postgres...")
conn = psycopg2.connect(**DB_CFG)
cur = conn.cursor()

print("Ensuring tables exist...")

cur.execute("""
CREATE TABLE IF NOT EXISTS first_half_basics (
    game_id INTEGER NOT NULL,
    game_date DATE NOT NULL,
    team VARCHAR(50) NOT NULL,
    opponent VARCHAR(50) NOT NULL,
    site CHAR(1) NOT NULL,                 -- 'H' or 'A'
    first_half_result CHAR(1) NOT NULL,    -- 'W', 'L', 'T'
    pts_scored_1 INTEGER NOT NULL,
    pts_allowed INTEGER NOT NULL,
    margin INTEGER NOT NULL,
    PRIMARY KEY (game_id, team)
);
""")

cur.execute("CREATE INDEX IF NOT EXISTS idx_first_half_basics_date ON first_half_basics (game_date);")
cur.execute("CREATE INDEX IF NOT EXISTS idx_first_half_basics_team ON first_half_basics (team);")

# Cursor progress table
cur.execute("""
CREATE TABLE IF NOT EXISTS api_progress_games_cursor (
    source VARCHAR(50) NOT NULL,
    season INTEGER NOT NULL,
    last_cursor BIGINT,               -- nullable; NULL means start
    updated_at TIMESTAMP NOT NULL DEFAULT NOW(),
    PRIMARY KEY (source, season)
);
""")

conn.commit()
print("✅ Tables ready.")
print(f"Starting throttle: {throttle_sec:.2f}s\n")

def get_last_cursor(season: int):
    cur.execute("""
        SELECT last_cursor
        FROM api_progress_games_cursor
        WHERE source='balldontlie_games' AND season=%s
    """, (season,))
    row = cur.fetchone()
    return row[0] if row else None

def set_last_cursor(season: int, cursor_val):
    cur.execute("""
        INSERT INTO api_progress_games_cursor (source, season, last_cursor, updated_at)
        VALUES ('balldontlie_games', %s, %s, NOW())
        ON CONFLICT (source, season)
        DO UPDATE SET last_cursor=EXCLUDED.last_cursor, updated_at=NOW();
    """, (season, cursor_val))

# ----------------------------
# LOAD
# ----------------------------
session = requests.Session()

for season in SEASONS:
    cursor_val = get_last_cursor(season)
    print(f"=== Season {season} (resuming cursor={cursor_val}) ===")

    batch_num = 0
    while True:
        params = {
            "per_page": PER_PAGE,
            "seasons[]": [season],
        }
        if cursor_val is not None:
            params["cursor"] = cursor_val

        r = safe_get(session, params)
        payload = r.json()
        games = payload.get("data", [])
        meta  = payload.get("meta", {}) or {}
        next_cursor = meta.get("next_cursor", None)

        if not games:
            print(f"✅ Season {season} complete.\n")
            conn.commit()
            break

        batch_num += 1
        print(f"Season {season} - Batch {batch_num}: {len(games)} games (cursor={cursor_val}, throttle={throttle_sec:.2f}s)")

        printed_sample = False
        batch_rows_attempted = 0

        for g in games:
            # Quarter scoring fields are on the game object directly
            home_q1 = g.get("home_q1")
            home_q2 = g.get("home_q2")
            vis_q1  = g.get("visitor_q1")
            vis_q2  = g.get("visitor_q2")

            if None in (home_q1, home_q2, vis_q1, vis_q2):
                continue

            try:
                home_1h = int(home_q1) + int(home_q2)
                away_1h = int(vis_q1) + int(vis_q2)
            except (ValueError, TypeError):
                continue

            home = g["home_team"]["abbreviation"]
            away = g["visitor_team"]["abbreviation"]

            game_date = (g.get("date") or "")[:10]
            game_id = g["id"]

            team_rows = [
                (game_id, game_date, home, away, "H", home_1h, away_1h),
                (game_id, game_date, away, home, "A", away_1h, home_1h),
            ]

            for game_id, gd, team, opp, site, pts_for, pts_against in team_rows:
                margin = pts_for - pts_against
                result = "W" if margin > 0 else "L" if margin < 0 else "T"

                cur.execute("""
                    INSERT INTO first_half_basics (
                        game_id, game_date, team, opponent, site,
                        first_half_result, pts_scored_1, pts_allowed, margin
                    )
                    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
                    ON CONFLICT (game_id, team) DO NOTHING;
                """, (game_id, gd, team, opp, site, result, pts_for, pts_against, margin))

                batch_rows_attempted += 1

                # Print only when a NEW row was actually inserted
                if (not printed_sample) and (cur.rowcount == 1):
                    pretty_print_row(gd, team, opp, site, pts_for, pts_against, margin, result)
                    printed_sample = True

        conn.commit()

        # Save progress AFTER successful commit
        set_last_cursor(season, next_cursor)
        conn.commit()

        print(f"  ✅ committed batch (attempted rows: {batch_rows_attempted}), next_cursor={next_cursor}\n")

        if next_cursor is None:
            print(f"✅ Season {season} complete (no next_cursor).\n")
            break

        cursor_val = next_cursor

cur.close()
conn.close()
print("✅ Postgres connection closed.")



Connecting to Postgres...
Ensuring tables exist...
✅ Tables ready.
Starting throttle: 4.00s

=== Season 2023 (resuming cursor=None) ===
Season 2023 - Batch 1: 50 games (cursor=None, throttle=4.00s)
  ✅ committed batch (attempted rows: 100), next_cursor=1037642

Season 2023 - Batch 2: 50 games (cursor=1037642, throttle=4.00s)
  👀 Sample inserted row:
    2023-10-30 | LAL vs ORL | Site=H | 1H: 54–58 | Margin=-4 | Result=L
  ✅ committed batch (attempted rows: 100), next_cursor=1037692

Season 2023 - Batch 3: 50 games (cursor=1037692, throttle=4.00s)
  👀 Sample inserted row:
    2023-11-06 | MIN vs BOS | Site=H | 1H: 52–55 | Margin=-3 | Result=L
  ✅ committed batch (attempted rows: 100), next_cursor=3277427

Season 2023 - Batch 4: 50 games (cursor=3277427, throttle=4.00s)
  👀 Sample inserted row:
    2023-11-14 | BKN vs ORL | Site=H | 1H: 57–60 | Margin=-3 | Result=L
  ✅ committed batch (attempted rows: 100), next_cursor=1037792

Season 2023 - Batch 5: 50 games (cursor=1037792, throttle=4.

## The below cell is the daily that update the raw data NBA first half scores

In [ ]:
import os
import time
import requests
import psycopg2
from datetime import date, timedelta
import random

# ----------------------------
# CONFIG
# ----------------------------
API_KEY = os.getenv("BALLDONTLIE_API_KEY")
BASE_URL = "https://api.balldontlie.io/v1/games"

DB_CFG = dict(
    dbname="nba_data",
    user="postgres",
    password=os.getenv("NBA_DB_PASSWORD"),
    host="localhost",
    port=5432
)

PER_PAGE = 100
HEADERS = {"Authorization": API_KEY, "Accept": "application/json"}

THROTTLE_SEC = 1.5  # lighter for small backfills

def safe_get(session, params, max_retries=10):
    for attempt in range(1, max_retries + 1):
        time.sleep(THROTTLE_SEC)

        r = session.get(BASE_URL, headers=HEADERS, params=params, timeout=30)

        if r.status_code == 429:
            wait = min(10 * attempt, 120)
            print(f"  ⚠️ 429 Rate limit. Retry {attempt}/{max_retries} in {wait}s")
            time.sleep(wait)
            continue

        if r.status_code in (500, 502, 503, 504):
            wait = min(2 ** attempt, 30) + random.uniform(0, 1.5)
            print(f"  ⚠️ HTTP {r.status_code}. Retry {attempt}/{max_retries} in {wait:.1f}s")
            time.sleep(wait)
            continue

        r.raise_for_status()
        return r

    raise RuntimeError(f"Failed after {max_retries} retries for params={params}")

# ----------------------------
# DB CONNECT
# ----------------------------
print("Connecting to Postgres...")
conn = psycopg2.connect(**DB_CFG)
cur = conn.cursor()

# What’s the last date we have?
cur.execute("SELECT MAX(game_date) FROM first_half_basics;")
last_date = cur.fetchone()[0]

if last_date is None:
    raise RuntimeError("first_half_basics is empty. Use your full-season loader first.")

start_date = last_date + timedelta(days=1)
end_date = date.today() - timedelta(days=1)  # up through yesterday (safe)

if start_date > end_date:
    print(f"✅ No backfill needed. first_half_basics already up-to-date through {last_date}.")
    cur.close()
    conn.close()
    raise SystemExit

print(f"📆 Backfilling from {start_date} through {end_date}\n")

# ----------------------------
# LOAD MISSING DATES
# ----------------------------
session = requests.Session()

total_inserted = 0
total_attempted = 0

d = start_date
while d <= end_date:
    ds = d.strftime("%Y-%m-%d")
    print(f"=== Fetching games for {ds} ===")

    params = {
        "per_page": PER_PAGE,
        "dates[]": ds,
    }

    r = safe_get(session, params)
    payload = r.json()
    games = payload.get("data", []) or []

    print(f"  Games returned: {len(games)}")

    inserted_this_day = 0
    attempted_this_day = 0

    for g in games:
        home_q1 = g.get("home_q1")
        home_q2 = g.get("home_q2")
        vis_q1  = g.get("visitor_q1")
        vis_q2  = g.get("visitor_q2")

        # If the API didn't provide quarter scoring, skip
        if None in (home_q1, home_q2, vis_q1, vis_q2):
            continue

        try:
            home_1h = int(home_q1) + int(home_q2)
            away_1h = int(vis_q1) + int(vis_q2)
        except (ValueError, TypeError):
            continue

        home = g["home_team"]["abbreviation"]
        away = g["visitor_team"]["abbreviation"]
        game_date = (g.get("date") or "")[:10]
        game_id = g["id"]

        team_rows = [
            (game_id, game_date, home, away, "H", home_1h, away_1h),
            (game_id, game_date, away, home, "A", away_1h, home_1h),
        ]

        for game_id, gd, team, opp, site, pts_for, pts_against in team_rows:
            margin = pts_for - pts_against
            result = "W" if margin > 0 else "L" if margin < 0 else "T"

            cur.execute("""
                INSERT INTO first_half_basics (
                    game_id, game_date, team, opponent, site,
                    first_half_result, pts_scored_1, pts_allowed, margin
                )
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
                ON CONFLICT (game_id, team) DO NOTHING;
            """, (game_id, gd, team, opp, site, result, pts_for, pts_against, margin))

            attempted_this_day += 1
            if cur.rowcount == 1:
                inserted_this_day += 1

    conn.commit()

    print(f"  ✅ {ds} committed. Inserted {inserted_this_day}/{attempted_this_day} attempted rows.\n")

    total_inserted += inserted_this_day
    total_attempted += attempted_this_day

    d += timedelta(days=1)

cur.close()
conn.close()

print(f"🏁 Backfill complete. Total inserted: {total_inserted} (attempted {total_attempted}).")


Connecting to Postgres...
📆 Backfilling from 2025-12-15 through 2025-12-17

=== Fetching games for 2025-12-15 ===
  Games returned: 5
  ✅ 2025-12-15 committed. Inserted 10/10 attempted rows.

=== Fetching games for 2025-12-16 ===
  Games returned: 1
  ✅ 2025-12-16 committed. Inserted 2/2 attempted rows.

=== Fetching games for 2025-12-17 ===
  Games returned: 2
  ✅ 2025-12-17 committed. Inserted 4/4 attempted rows.

🏁 Backfill complete. Total inserted: 16 (attempted 16).
